In [ ]:
"""
多臂老虎机（MAB）实现
不同于一般的强化学习过程，多臂老虎机问题没有使用状态及状态转移分布，只使用了动作空间和奖励函数
"""

#导入使用的库，numpy库用于数组和矩阵运算，matplotlib库用于绘图
import numpy as np
import matplotlib.pyplot as plt

#构建伯努利多臂老虎机类
class BernoulliBandit:
	"""输入k表示拉杆个数，决定了动作空间的大小"""
	def __init__(self, K):
	#初始化函数实现
		self.probs = np.random.uniform(size = K)
		#随机生成K个0～1均匀分布的随机数,作为拉动每根拉杆的获奖概率
		#uniform方法接受3个参数，low最小值，high最大值和size输出形状
		self.best_idx = np.argmax(self.probs)
		#argmax方法返回数组中最大值所在的下标，从而得到获奖概率最高的拉杆id
		self.best_prob = self.probs[self.best_idx]
		#通过下标索引得到最高获奖概率
		self.K = K
		#输入K即为拉杆个数

	def step(self, k):
	#玩家作出决策及奖励反馈函数
	#当玩家选择k号拉杆后，依据获奖概率返回1（获奖）和0（未获奖）
	#随机在0～1间均匀分布采样，与获奖概率进行比较
		if np.random.rand() < self.probs[k]:
		#利用random.rand()方法生成0～1间随机数，与对应k号杆获奖概率进行对比
		#random.rand()方法只能用于生成0～1间的均匀分布随机数，接受参数size输出形状
		#random.randn()方法哟更浓郁生成标准正态分布随机数
		#当随机数小于获奖概率时认为获奖，返回1
			return 1
		else:
		#当随机数大于获奖概率时认为未获奖，返回0
			return 0

np.random.seed(1)	#设定随机种子数，使实验具有可重复性
K = 10
bandit_10_arm = BernoulliBandit(K)	#生成10臂老虎机实例
print("随机生成了一个%d臂伯努利老虎机" % K)
print("获奖概率最大的拉杆为%d号，其获奖概率为%.4f" % (bandit_10_arm.best_idx, bandit_10_arm.best_prob))
#调用两个成员函数获取实例信息
for i in range(K):
#打印每根拉杆获奖概率，便于验证
    print("第%d号拉杆获奖概率为%.4f" % (i, bandit_10_arm.probs[i]))

"""
构键Solver类实现以下功能：
1. 根据策略选择动作
2. 根据动作获取奖励
3. 更新期望奖励估值
4. 更新累计懊悔和期望
前3条放在run_one_step()函数中实现，第4条在主循环run()函数中实现
"""
class Solver:
    """构建求解器Solver类，搭建多臂老虎机算法基本框架"""
    def __init__(self, bandit):
    #初始化函数实现
        self.bandit = bandit
        self.counts = np.zeros(self.bandit.K)
        #构建counts数组用于计数每根拉杆尝试次数，初始化为全零，传入参数为拉杆总数，保持维数对齐
        self.regret = 0
        #用于记录当前步的累积懊悔值。初始化为0
        self.actions = []
        #维护一个actions列表，用于记录每一步的动作
        self.regrets = []
        #维护一个regrets列表，记录每一步的累积懊悔

    def update_regret(self, k):
    #更新懊悔值函数实现。计算累计懊悔并保存
    #传入参数参数k为本次动作选择的拉杆的编号，表明了选择的动作，用于计算该步产生的懊悔值
        self.regret += self.bandit.best_prob - self.bandit.probs[k]
        #右侧表达式说明了懊悔的计算过程，通过+=运算使左值产生累计效果
        self.regrets.append(self.regret)
        #将计算出的该步累计懊悔值追加到regrets列表中，实现记录功能
        #append()方法会返回新数组，不会修改原数组

    def run_one_step(self):
    #一步动作更新函数
    #返回当前动作选择哪一根拉杆，有每个具体的策略实现
        raise NotImplementedError
        #这里使用raise NotImplementedError表示该方法需要在子类中具体实现

    def run(self, num_steps):
    #主循环函数
    #运行一定次数，传入参数num_steps为总运行次数
        for _ in range(num_steps):
            k = self.run_one_step()
            #
            self.counts[k] += 1
            #计数器数组中第k号拉杆选中次数+1
            self.actions.append(k)
            #追加动作列表本次动作选择为拉动k号拉杆
            self.update_regret(k)
            #计算当前步拉动k号拉杆的累积懊悔

"""
epsilon-greedy算法实现
"""
class EpsilonGreedy(Solver):
    """Epsilon贪婪算法，继承Solver类"""
    def __init__(self, bandit, epsilon = 0.01, init_prob = 1.0):
    #初始化函数实现，接受参数
        super(EpsilonGreedy, self).__init__(bandit)
        #继承初始化，super的用法为super(子类名, self).__init__(父类初始化参数)
        self.epsilon = epsilon
        #该参数用于控制随机采样概率
        self.estimates = np.array([init_prob] * self.bandit.K)
        #该数组用于记录每根拉杆的期望奖励估值，初始化为全1，便于后续更新
        
    def run_one_step(self):
        #一步动作更新函数
        #返回当前动作选择哪一根拉杆
        if np.random.random() < self.epsilon:
            #random.random()与random.rand()方法的区别是前者生成0～1间的浮点数，后者生成0～1间的整数
            #以epsilon概率进行随机探索
            k = np.random.randint(0, self.bandit.K)
            #随机生成0～K-1间的整数，作为动作选择，拉动k号拉杆
            #random.randint()方法用于生成区间内随机整数，接受参数为low(包含), high(不包含), size(输出形状)
        else:
            #以1-epsilon概率进行贪婪选择
            k = np.argmax(self.estimates)
            #返回估值最大的拉杆编号
        r = self.bandit.step(k)
        #调用step函数获取奖励，具体定义在BernoulliBandit类中
        self.estimates[k] += 1. / (self.counts[k] + 1) * (r - self.estimates[k])
        #使用增量式更新公式更新k号拉杆的期望奖励估值
        return k
        #返回动作选择，之后加入actions列表
    
def plot_results(solvers, solver_names):
    """
    生成累计懊悔随时间变化的图像；
    输入solvers是一个列表，列表中的每个元素是一种特定的策略；
    而输入solver_names也是一个列表，列表中的每个元素是对应策略的名称
    """
    #绘制累计懊悔随时间变化的图像函数实现
    for idx, solver in enumerate(solvers):
        #遍历所有策略，其中idx为索引，solver为策略实例，符合列表存储
        #enumerate()函数用于遍历索引和元素本身
        plt.plot(solver.regrets, label = solver_names[idx])
        #使用matplotlib库绘制累计懊悔随时间变化的图像
        #使用plot函数绘制累计懊悔随时间变化的图像，传入参数为regrets列表和标签
        #其中solver.regrets为策略实例的regrets属性，即累积懊悔列表
    plt.xlabel('Time steps')
    #设置x轴标签为时间步数
    plt.ylabel('Cumulative regret')
    #设置y轴标签为累积懊悔
    plt.title('%d-armed bandit' % solvers[0].bandit.K)
    #设置标题为%d-armed bandit，其中%d为拉杆数
    #在python中单引号和双引号基本一致
    plt.legend()
    #显示图例
    plt.show()
    #显示图像

np.random.seed(1)
#设置随机数种子，保证实验可重复
#这里的随机数与前文的随机种子不同，互不影响
#可以进行实验比较
epsilon_greedy_solver = EpsilonGreedy(bandit_10_arm, epsilon=0.01)
#实例化epsilon-贪婪算法求解器
epsilon_greedy_solver.run(5000)
#运行5000次
print('epsilon-贪婪算法的累计懊悔为：', epsilon_greedy_solver.regret)
#打印epsilon-贪婪算法的累计懊悔
plot_results([epsilon_greedy_solver], ["EpsilonGreedy"])
#绘制epsilon-贪婪算法的累计懊悔随时间变化的图像